# Baseline Recommenders

Notebook ini membangun dan membandingkan baseline **Popularity**, **Item-CF**, dan **TF-IDF CBF**. Model belajar dari `interactions_train.csv`; validasi warm hanya dipakai untuk evaluasi.

Protokol: user dari `evaluable_users_validation_warm.csv`, rating >= 4 sebagai relevan, kandidat dari `candidate_items_warm.csv`, dan item yang sudah pernah berinteraksi saat train dikeluarkan dari rekomendasi.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Buka notebook dari folder proyek agar folder src/ dan data tersedia.")
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_project_data
from src.evaluation import build_relevant_items, evaluate_ranking
from src.recommenders import fit_popularity, fit_item_cf, fit_tfidf_cbf, recommend_popularity, recommend_from_model

TOP_N = 200
K = 10
data = load_project_data(PROJECT_ROOT)
train = data.train
items = data.items
candidates = data.candidates
eval_users = data.evaluable_users
validation = data.validation_warm
relevant = build_relevant_items(validation, eval_users, positive_threshold=4)

print(f"Train: {len(train):,} interaksi | {train.user_id.nunique():,} user | {train.item_id.nunique():,} item")
print(f"Kandidat warm: {len(candidates):,} | User evaluasi: {len(eval_users):,}")
print(f"User dengan target relevan: {len(relevant):,}")

Train: 427,957 interaksi | 54,483 user | 22,753 item
Kandidat warm: 22,753 | User evaluasi: 11,659
User dengan target relevan: 11,659


## 1. Popularity

Popularitas dihitung dari jumlah interaksi positif (rating >= 4) pada data train. Ini menjadi pembanding tanpa personalisasi.

Untuk Item-CF dan CBF, item yang memiliki skor nol dipakai sebagai pengisi ranking dengan urutan popularitas. Ini menjaga daftar rekomendasi cukup panjang dan membuat fallback-nya eksplisit.

In [ ]:
popularity = fit_popularity(train, positive_threshold=4)
seen_by_user = {
    user: set(group.item_id.astype(str))
    for user, group in train.groupby("user_id", sort=False)
}
pop_recs = recommend_popularity(
    eval_users, candidates, popularity, seen_by_user, top_n=TOP_N
)
pop_metrics = evaluate_ranking(pop_recs, relevant, eval_users, k=K)
print(f"Jumlah user dievaluasi: {pop_metrics['users_evaluated']:,}")
pd.DataFrame(
    [{metric: pop_metrics[metric] for metric in ["recall@10", "ndcg@10"]}],
    index=["Popularity"],
)

Jumlah user dievaluasi: 11,659


,recall@10,ndcg@10
Popularity,0.013499,0.009222


### Melihat rekomendasi Popularity

Tabel pertama menunjukkan item yang paling sering mendapat rating positif di train. Tabel kedua menunjukkan 10 rekomendasi untuk satu user evaluasi setelah item yang sudah pernah ia lihat dikeluarkan.

In [ ]:
catalog = items[["item_id", "title"]].copy()
catalog["item_id"] = catalog["item_id"].astype(str)

top_popular = (
    popularity.head(10)
    .rename_axis("item_id")
    .reset_index(name="positive_train_interactions")
    .merge(catalog, on="item_id", how="left")
)
display(top_popular[["item_id", "title", "positive_train_interactions"]])

sample_user = eval_users[0]
sample_recommendations = (
    pd.DataFrame(pop_recs[sample_user][:10], columns=["item_id", "positive_train_interactions"])
    .merge(catalog, on="item_id", how="left")
)
print(f"Contoh rekomendasi untuk user: {sample_user}")
display(sample_recommendations[["item_id", "title", "positive_train_interactions"]])

,item_id,title,positive_train_interactions
0,B09857JRP2,GLS Audio Instrument Cable - Amp Cord for Bass...,2175
1,B0BCK6L7S5,"Donner Guitar Delay Pedal, Yellow Fall Analog ...",1441
2,B0BSGM6CQ9,Ernie Ball Mondo Slinky Nickelwound Electric G...,1404
3,B0BPJ4Q6FJ,D'Addario Guitar Strings - XL Nickel Electric ...,1271
4,B0BTC9YJ2W,D'Addario Guitar Strings - Phosphor Bronze Aco...,1185
5,B004XNK7AI,"Snark SN-5 Tuner for Guitar, Bass and Violin (...",1119
6,B0BHSG5C3G,"Snark SN5X Clip-On Tuner for Guitar, Bass & Vi...",1031
7,B08R5GM6YB,"D'Addario Pro-Winder - Guitar String Winder, G...",1002
8,B0C67HCGBR,GLS Audio 12ft Mic Cable Patch Cords - XLR Mal...,917
9,B0BPLFP5P6,D'Addario Accessories Guitar Tuner - Micro Hea...,895


Contoh rekomendasi untuk user: AE22B7BMXT7GAGY3XLHICMESNAPA


,item_id,title,positive_train_interactions
0,B09857JRP2,GLS Audio Instrument Cable - Amp Cord for Bass...,2175.0
1,B0BCK6L7S5,"Donner Guitar Delay Pedal, Yellow Fall Analog ...",1441.0
2,B0BSGM6CQ9,Ernie Ball Mondo Slinky Nickelwound Electric G...,1404.0
3,B0BPJ4Q6FJ,D'Addario Guitar Strings - XL Nickel Electric ...,1271.0
4,B0BTC9YJ2W,D'Addario Guitar Strings - Phosphor Bronze Aco...,1185.0
5,B004XNK7AI,"Snark SN-5 Tuner for Guitar, Bass and Violin (...",1119.0
6,B0BHSG5C3G,"Snark SN5X Clip-On Tuner for Guitar, Bass & Vi...",1031.0
7,B08R5GM6YB,"D'Addario Pro-Winder - Guitar String Winder, G...",1002.0
8,B0C67HCGBR,GLS Audio 12ft Mic Cable Patch Cords - XLR Mal...,917.0
9,B0BPLFP5P6,D'Addario Accessories Guitar Tuner - Micro Hea...,895.0


## 2. Item-CF

Item-CF membentuk kemiripan cosine dari item yang sama-sama mendapat rating >= 4 pada data train. Rekomendasi user dihitung dari item positif yang pernah dinilai user tersebut.

In [ ]:
item_cf = fit_item_cf(train, candidates, top_k=100, positive_threshold=4)
itemcf_recs = recommend_from_model(item_cf, eval_users, candidates, popularity, top_n=TOP_N)
itemcf_metrics = evaluate_ranking(itemcf_recs, relevant, eval_users, k=K)
itemcf_metrics

{'users_evaluated': 11659,
 'recall@10': 0.005885283512122505,
 'ndcg@10': 0.003350967498350444}

### Melihat rekomendasi Item-CF

Item-CF memakai pola item yang sama-sama disukai user pada data train. Skor Item-CF menjumlahkan kemiripan item dari riwayat positif user. Jika skor model nol, item populer dipakai sebagai pengisi daftar.

In [ ]:
sample_itemcf = (
    pd.DataFrame(itemcf_recs[sample_user][:10], columns=["item_id", "itemcf_score"])
    .merge(catalog, on="item_id", how="left")
)
display(sample_itemcf[["item_id", "title", "itemcf_score"]])

,item_id,title,itemcf_score
0,B09XXG98XF,"Korg, Metronome (KDM3WH)",0.500000
1,B07JLFRBN8,Alnicov Set of 3 Alnico V Single-Coil-Sized Hu...,0.288675
2,B08Y1NRCWW,Gator Frameworks Utility Table Top and X Style...,0.277350
3,B08BQJYNCQ,Premium Guitar Spanner Wrench with thin Foam s...,0.250000
4,B08GS84HXJ,Donner DPA-100 Guitar Pedal Power Supply Adapt...,0.235702
5,B01L1GCETG,Vizcaya Black Rubber Clarinet Thumb Rest Cushi...,0.235702
6,B00HYMBMEU,Aquila 107U Tenor Ukulele Strings Set,0.235702
7,B0BZ5BMQ5P,"Buckle-Down DC Comics Guitar Strap, Superman S...",0.223607
8,B00LA10IUU,Neweights Cotton Rainbow Color Ukulele Strap w...,0.213201
9,B0C2PY3PJV,"JYX Portable Bluetooth Karaoke Machine, Microp...",0.208514


## 3. TF-IDF Content-Based Filtering

TF-IDF dibangun dari `content_text_basic`. Model membentuk kemiripan konten antar-item, lalu mempersonalisasi ranking dari item positif dalam riwayat train user.

In [ ]:
tfidf_cbf = fit_tfidf_cbf(train, items, candidates, text_column="content_text_basic", top_k=100, positive_threshold=4)
cbf_recs = recommend_from_model(tfidf_cbf, eval_users, candidates, popularity, top_n=TOP_N)
cbf_metrics = evaluate_ranking(cbf_recs, relevant, eval_users, k=K)
cbf_metrics

{'users_evaluated': 11659,
 'recall@10': 0.012233976372245561,
 'ndcg@10': 0.0074920722856896835}

### Melihat hasil pemecahan teks oleh TF-IDF

`TfidfVectorizer` melakukan tokenisasi secara otomatis saat `fit_transform()`. Cell ini memperlihatkan kata tunggal dan pasangan kata yang dihasilkan untuk satu produk.

In [ ]:
sample_item_id = str(candidates[0])
sample_text = items.loc[
    items["item_id"].astype(str) == sample_item_id, "content_text_basic"
].fillna("").iloc[0]
text_analyzer = tfidf_cbf["vectorizer"].build_analyzer()
print("Teks produk:", sample_text[:250], "...")
print("Hasil tokenisasi:", text_analyzer(sample_text)[:30])

Teks produk: Guiros Wooden Percussion 3 Piece Set 3 Inch Wooden Frog, 4 Inch Wooden Cricket and 3.5 Inches Wooden Owl. Musical Instruments WADSUWAN SHOP Musical Instruments Drums & Percussion Hand Percussion Folk & World Latin Percussion Guiros Frog - Rubbing its ...
Hasil tokenisasi: ['guiros', 'wooden', 'percussion', 'piece', 'set', 'inch', 'wooden', 'frog', 'inch', 'wooden', 'cricket', 'inches', 'wooden', 'owl', 'musical', 'instruments', 'wadsuwan', 'shop', 'musical', 'instruments', 'drums', 'percussion', 'hand', 'percussion', 'folk', 'world', 'latin', 'percussion', 'guiros', 'frog']


## Sanity check pada subset validation

Pemeriksaan manual kecil ini mengambil 5 user secara tetap (seed 42) dari user evaluasi warm. Kita lihat target relevan validation dan lima rekomendasi teratas dari Popularity, Item-CF, serta TF-IDF CBF. Kolom `target_validation` hanya menandai apakah item rekomendasi kebetulan ada di target validation; ini untuk inspeksi, bukan bagian dari training. Skor tiap metode punya skala berbeda.

In [ ]:
SANITY_USER_COUNT = 5
SANITY_TOP_N = 5
sanity_users = (
    pd.Series(sorted(relevant), dtype="string")
    .sample(n=min(SANITY_USER_COUNT, len(relevant)), random_state=42)
    .astype(str)
    .tolist()
)

sanity_targets = []
sanity_rankings = {
    "Popularity": pop_recs,
    "Item-CF": itemcf_recs,
    "TF-IDF CBF": cbf_recs,
}
for user in sanity_users:
    target_ids = relevant.get(user, set())
    target_titles = catalog[catalog["item_id"].isin(target_ids)][["item_id", "title"]]
    for row in target_titles.itertuples(index=False):
        sanity_targets.append({
            "user_id": user,
            "validation_target_item": row.item_id,
            "title": row.title,
        })

print(f"Target positif pada validation untuk {len(sanity_users)} user contoh:")
display(pd.DataFrame(sanity_targets))

sanity_rows = []
for user in sanity_users:
    target_ids = relevant.get(user, set())
    for method, rankings in sanity_rankings.items():
        for rank, (item_id, score) in enumerate(rankings.get(user, [])[:SANITY_TOP_N], start=1):
            title_rows = catalog.loc[catalog["item_id"] == item_id, "title"]
            sanity_rows.append({
                "user_id": user,
                "metode": method,
                "rank": rank,
                "item_id": item_id,
                "title": title_rows.iloc[0] if not title_rows.empty else "(judul tidak tersedia)",
                "score": score,
                "target_validation": item_id in target_ids,
            })

print(f"Top-{SANITY_TOP_N} rekomendasi per metode untuk {len(sanity_users)} user:")
display(pd.DataFrame(sanity_rows))

Target positif pada validation untuk 5 user contoh:


,user_id,validation_target_item,title
0,AFDK7KSL3FXCCTUCVPNQCINQBOUQ,B09G5QS1VD,Ernie Ball Beefy Slinky Cobalt Electric Guitar...
1,AFYBYHKCRE724MU26GDAKE2Q2SJA,B09R6W3B2Y,Yamaha FC7 Volume Expression Pedal for Keyboards
2,AH2O3QU4KENCEUNWEIUG6O2IXYLQ,B005CZIJJC,Latin Percussion LP160NY-K Tambourine and Cowb...
3,AH2O3QU4KENCEUNWEIUG6O2IXYLQ,B07GY7828V,Hipshot KickAss High-mass Bass Bridge for 5-bo...
4,AH5VDE2Z5X2ZBRZ3IYMSCVW7ZWFQ,B09RWRQNR9,"LEKATO Looper Pedal, Looper Pedals with SYNC &..."
5,AH5VDE2Z5X2ZBRZ3IYMSCVW7ZWFQ,B0B68Z7PG6,pigtone Overdrive Distortion Guitar Effect Ped...
6,AH5VDE2Z5X2ZBRZ3IYMSCVW7ZWFQ,B0BKL6VQYG,Hricane Concert Ukulele Spalted Maple 23 Inch ...
7,AH5VDE2Z5X2ZBRZ3IYMSCVW7ZWFQ,B0BX8Z1RTF,EX Signal Line Select A/B Box Passive Circuit ...
8,AG7YJ74TSTCE4JDFJM6I6GNGOHSA,B08LVQKMKZ,Musiclily Basic Mini Electric Guitar Effects P...


Top-5 rekomendasi per metode untuk 5 user:


,user_id,metode,rank,item_id,title,score,target_validation
0,AFDK7KSL3FXCCTUCVPNQCINQBOUQ,Popularity,1,B09857JRP2,GLS Audio Instrument Cable - Amp Cord for Bass...,2175.000000,False
1,AFDK7KSL3FXCCTUCVPNQCINQBOUQ,Popularity,2,B0BCK6L7S5,"Donner Guitar Delay Pedal, Yellow Fall Analog ...",1441.000000,False
2,AFDK7KSL3FXCCTUCVPNQCINQBOUQ,Popularity,3,B0BSGM6CQ9,Ernie Ball Mondo Slinky Nickelwound Electric G...,1404.000000,False
3,AFDK7KSL3FXCCTUCVPNQCINQBOUQ,Popularity,4,B0BPJ4Q6FJ,D'Addario Guitar Strings - XL Nickel Electric ...,1271.000000,False
4,AFDK7KSL3FXCCTUCVPNQCINQBOUQ,Popularity,5,B0BTC9YJ2W,D'Addario Guitar Strings - Phosphor Bronze Aco...,1185.000000,False
...,...,...,...,...,...,...,...
70,AG7YJ74TSTCE4JDFJM6I6GNGOHSA,TF-IDF CBF,1,B083ZW5R47,Wilkinson Low Gauss Hot Rail Ceramic Single Co...,0.839413,False
71,AG7YJ74TSTCE4JDFJM6I6GNGOHSA,TF-IDF CBF,2,B083ZW6B4Y,Wilkinson Low Gauss Hot Rail Ceramic Single Co...,0.825981,False
72,AG7YJ74TSTCE4JDFJM6I6GNGOHSA,TF-IDF CBF,3,B08R3FDHV9,"Lutiore Guitar Cable 10ft, 2 Pack Electric Ins...",0.631682,False
73,AG7YJ74TSTCE4JDFJM6I6GNGOHSA,TF-IDF CBF,4,B0B8LQFN3C,D'Addario Accessories Guitar Strap - Guitar Ac...,0.624803,False


## 4. Perbandingan awal

Nilai di bawah adalah macro-average pada user evaluasi yang memiliki setidaknya satu item relevan. Hasil validation dipakai untuk pengembangan dan pemilihan parameter; jangan dianggap sebagai hasil test akhir.

In [ ]:
baseline_results = pd.DataFrame([
    {"Metode": "Popularity", "recall@10": pop_metrics["recall@10"], "ndcg@10": pop_metrics["ndcg@10"]},
    {"Metode": "Item-CF", "recall@10": itemcf_metrics["recall@10"], "ndcg@10": itemcf_metrics["ndcg@10"]},
    {"Metode": "TF-IDF CBF", "recall@10": cbf_metrics["recall@10"], "ndcg@10": cbf_metrics["ndcg@10"]},
]).set_index("Metode")
baseline_results.sort_values("ndcg@10", ascending=False)

,recall@10,ndcg@10
Metode,,
Popularity,0.013499,0.009222
TF-IDF CBF,0.012234,0.007492
Item-CF,0.005885,0.003351


### Catatan

Notebook ini sengaja tidak membaca test set. Implementasi fusi Weighted Score dan RRF berada di `03_hybrid_fusion.ipynb`.